# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Selected Lane:** **Refresh / Content Opportunity Scoring**

*Why this lane?* In enterprise search engine optimization, content decay represents an invisible, compounding revenue leak. Editorial teams cannot manually audit tens of thousands of published articles every month. Rather than applying blunt, static heuristic rules (such as reviewing only top-impression URLs or arbitrarily updating articles after 6 months), machine learning enables data-driven, multi-signal prioritization. By coupling search visibility decay signals with engagement and position metrics, we can score and rank existing content into an actionable editorial refresh queue with granular reason codes.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)
print(f"Loaded dataset successfully: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Number of distinct clients represented: {df['client_id'].nunique()}")
print(f"Content types present: {df['content_type'].value_counts().to_dict()}")

Loaded dataset successfully: 30,000 rows x 44 columns
Number of distinct clients represented: 32
Content types present: {'keyword article': 27207, 'feedly article': 2096, 'comparison article': 697}


## 2. The question: decision, action, cost of a wrong call

**Research Question:** *Which published content assets exhibit the highest probability of search performance decay, and how can editorial teams systematically prioritize them for updates before organic visibility collapses?*

- **Decision Supported:** Deciding whether a given page should receive an editorial refresh (fact update, section expansion), a CTR metadata overhaul (title tag and meta description rewrite), continuous monitoring, or no action.
- **Unit of Analysis:** A single published content item (`content_id`) belonging to an enterprise client (`client_id`) evaluated over a trailing 90-day measurement window.
- **Action Taken:** The SEO/Editorial team pulls the top 50 ranked pages from the weekly refresh queue and assigns them to content specialists with concrete reason codes (`declining_with_demand`, `low_ctr_visible_page`, etc.).
- **Cost of a Wrong Call:**
  - *False Positive (recommending a stable page):* Wastes 3–6 editorial hours rewriting content that was already ranking well, risking ranking disruption.
  - *False Negative (missing a decaying page):* High-opportunity content loses ranking silently, forfeiting organic search impressions, qualified sessions, and conversions to competitors.

In [2]:
# Verify distribution of trends and the cost profile
trend_counts = df['trend_direction'].value_counts()
trend_pcts = df['trend_direction'].value_counts(normalize=True) * 100

summary_df = pd.DataFrame({'Count': trend_counts, 'Percentage': trend_pcts})
print("Distribution of 90-day search trend directions:")
print(summary_df)

# Imbalance check: declining items represent the majority class
decline_base_rate = (df['trend_direction'] == 'down').mean()
print(f"\nTarget Base Rate (is_declining): {decline_base_rate:.3%}")

Distribution of 90-day search trend directions:
                 Count  Percentage
trend_direction                   
down             16262   54.206667
stable            5962   19.873333
up                4388   14.626667
new               2236    7.453333
flat              1152    3.840000

Target Base Rate (is_declining): 54.207%


## 3. Quick look at the data (2-3 real numbers)

Examining the FlyRank 30,000-row pseudonymized search dataset reveals key empirical benchmarks:
1. **Total Visibility Volume:** Across 32 clients, the dataset covers **44,834,167 impressions** and **339,088 clicks**, with an aggregate CTR of **0.756%**.
2. **Prevalence of Decline:** **16,262 of 30,000 pages (54.21%)** experienced negative search traffic trajectory over the trailing 90-day window (`trend_direction == 'down'`).
3. **Position Skew:** Over 68% of pages rank outside Page 1 (`avg_position > 10`), while 1,205 pages have `avg_position == 0` (indicating zero search impressions during the period).
4. **Content Age:** Content has a median age of **423 days**, with a long tail extending past 1,000 days without structural updates.

In [3]:
total_impressions = df['impressions_90d'].sum()
total_clicks = df['clicks_90d'].sum()
overall_ctr = (total_clicks / total_impressions) * 100 if total_impressions > 0 else 0
median_age = df['content_age_days'].median()
zero_pos_count = (df['avg_position'] == 0).sum()

print(f"Total Impressions (90d): {total_impressions:,}")
print(f"Total Clicks (90d):       {total_clicks:,}")
print(f"Aggregate CTR:            {overall_ctr:.3f}%")
print(f"Median Content Age:       {median_age:.1f} days")
print(f"Zero-position rows:       {zero_pos_count:,} ({zero_pos_count/len(df):.2%})")

Total Impressions (90d): 156,010,989
Total Clicks (90d):       482,920
Aggregate CTR:            0.310%
Median Content Age:       236.0 days
Zero-position rows:       1,205 (4.02%)


## 4. Careful words: what I can and can't claim

Scientific integrity and disciplined communication are paramount when analyzing search intelligence data:

**What This Project CAN Claim:**
- We observe empirical associations between engagement, position stability, age, and 90-day search trend directions across a 32-client sample.
- We provide a validated decision-support ranking system that surfaces decaying content with measured precision (e.g. Precision@50) superior to static heuristic rules.
- We produce transparent reason codes that explain the operational signals behind each recommendation.

**What This Project CANNOT Claim:**
- We do **not** claim to have reverse-engineered Google's proprietary search ranking algorithm.
- We do **not** claim causal proof that performing an editorial update will automatically restore lost rankings (ranking depends on external competition, search intent shifts, and search engine algorithm updates).
- We do **not** make claims regarding specific real-world client domains, proprietary query terms, or confidential client identities.

In [4]:
# Verify data privacy and safety boundaries
assert 'domain' not in df.columns, "Data leak: domain should not be present!"
assert 'query' not in df.columns, "Data leak: raw query strings should not be present!"
assert 'client_name' not in df.columns, "Data leak: client_name should not be present!"
print("Safety Audit Passed: No client names, raw domains, or query strings found in dataset.")

Safety Audit Passed: No client names, raw domains, or query strings found in dataset.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.